# finance / household / costs

Annual growth rate of household costs to 2070 under a low, base and high scenario.

| Term | In This Notebook |
|---|---|
| **Intent** | Give one annual cost growth rate per scenario to inflate spending to 2070. Placeholder: the base is the mean observed cost growth and the low is the mean inflation, with the high mirroring the structural premium of cost growth over inflation |
| **Project Flavour** | `pyfunc`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten years of inflation and cost growth, synthetic until the real series are loaded; `validation` five further years; `backtest` ten more, held back for acceptance; `golden` the exact scenarios that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/costs"
MODEL_NAME = "finance_household_costs"
TARGETS = [{"kind": "served"}]
TRAINING = "household_costs_training_2026-11-05_2026-11-05"
VALIDATION = "household_costs_validation_2026-11-05_2026-11-05"
BACKTEST = "household_costs_backtest_2026-11-05_2026-11-05"
GOLDEN = "household_costs_golden"
INPUTS = ["finance_household_costs_scenario"]
OUTPUTS = ["finance_household_costs_growth_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic around the two long run means, 3.24 percent inflation and 4.83 percent cost growth, so no query runs.

In [ ]:
INFLATION_DEVIATIONS = [-1.2, 0.8, -0.4, 1.0, -0.6, 0.2, 0.6, -1.0, 0.4, 0.2]
GROWTH_DEVIATIONS = [0.5, -0.7, 0.3, 0.9, -0.4, -0.8, 0.6, 0.2, -0.3, -0.3]


def observed(deviations):
    return {INPUTS[0]: [1.0] * len(deviations), OUTPUTS[0]: [round(4.83 + deviation, 4) for deviation in deviations]}


training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {
        "year": list(range(2016, 2026)),
        "inflation_pct": [round(3.24 + deviation, 4) for deviation in INFLATION_DEVIATIONS],
        "cost_growth_pct": [round(4.83 + deviation, 4) for deviation in GROWTH_DEVIATIONS],
    },
)
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: observed([0.2, -0.2, 0.4, -0.4, 0.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: observed(GROWTH_DEVIATIONS[::-1]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the mean inflation and the mean cost growth of `training` only (R6). The low scenario is the inflation mean, the base the cost growth mean and the high mirrors the premium of the base over the low. The premium is the structural drivers' stand-in until they are modelled (see the plan).

In [ ]:
INFLATION = round(float(training_pl["inflation_pct"].mean()), 2)
GROWTH = round(float(training_pl["cost_growth_pct"].mean()), 2)
PARAMS = {"low": INFLATION, "base": GROWTH, "high": round(2 * GROWTH - INFLATION, 2)}
print("fitted", PARAMS)
assert PARAMS == {"low": 3.24, "base": 4.83, "high": 6.42}, "the fit did not recover the long run means"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_household_costs_scenario"
OUTPUT = "finance_household_costs_growth_pct"
GROWTH = ({low!r}, {base!r}, {high!r})


class Costs(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [GROWTH[min(max(int(round(scenario)), 0), 2)] for scenario in model_input[INPUT]]}})


set_model(Costs())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_household_costs_scenario", "none", "Scenario, 0 low, 1 base, 2 high."),
    ],
    [
        signature_output("finance_household_costs_growth_pct", "percent", "Annual growth rate of costs for the scenario, percent per year.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 1.0
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [0.0, 1.0, 2.0, 1.4, 1.6, -1.0, 3.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual series around the long run inflation and cost growth means, training split",
        [training_path, golden_path],
        "mean inflation and mean cost growth per scenario",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the three scenarios and out of range ones",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)